In [ ]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tokenizer = tiktoken.get_encoding("gpt2")

text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)
# region  [token encoding]
# TODO: 인코딩 메서드명을 채워 text를 토큰 ID로 변환하세요.
# 힌트: allowed_special 인자는 그대로 두고 메서드 이름만 채우면 됩니다.
# 힌트: allowed_special 인자는 그대로 두고 메서드 이름만 채우면 됩니다.
# 예시: sample_ids = tokenizer.encode("Hi <|endoftext|>", allowed_special={"<|endoftext|>"})
integers = tokenizer.????(text, allowed_special={"<|endoftext|>"})
# endregion
print(integers)

# region  [token decoding]
# TODO: 디코딩 메서드명을 채워 토큰 ID를 문자열로 복원하세요.
# 힌트: integers를 입력으로 사용합니다.
# 힌트: integers를 입력으로 사용합니다.
# 예시: sample_text = tokenizer.decode(sample_ids)
strings = tokenizer.????(integers)
# endregion
print(strings)

tiktoken version: 0.12.0
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [ ]:
import os
import requests

os.makedirs("datas", exist_ok=True)
if not os.path.exists("datas/the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "datas/the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)
with open("datas/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

enc_sample = enc_text[50:]

context_size = 4

# TODO: y 시퀀스가 x보다 한 칸 오른쪽으로 미루도록 인덱스를 채우세요.
# 힌트: 시작과 끝 인덱스를 각각 +1 하면 next-token 타깃이 됩니다.
# 힌트: 시작 인덱스와 끝 인덱스를 각각 +1 이동하면 됩니다.
# 예시: x=[10,11,12,13]이면 y=[11,12,13,14]
x = enc_sample[:context_size]
y = enc_sample[????:context_size+?????]

print(f"x: {x}")
print(f"y:      {y}")

5145
x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [ ]:
### 중요: 이 코드는 GPT 모델 학습을 위한 데이터셋 클래스를 정의합니다

import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        """
        Args:
            txt (str): 학습할 전체 텍스트 데이터
            tokenizer: 텍스트를 토큰 ID로 변환해주는 토크나이저 (예: tiktoken)
            max_length (int): 모델이 한 번에 볼 수 있는 윈도우 크기 (입력 시퀀스 길이)
            stride (int): 윈도우를 이동시킬 간격 (데이터 중복 정도를 결정)
        """
        self.input_ids = []
        self.target_ids = []

        # 1. 전체 텍스트 토큰화
        # 텍스트를 정수 리스트(token_ids)로 변환합니다.
        # <|endoftext|> 같은 특수 토큰도 허용하여 인코딩합니다.
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 데이터가 너무 짧으면 학습할 수 없으므로 최소 길이를 확인합니다.
        assert len(token_ids) > max_length, "토큰화된 입력의 개수는 적어도 max_length+1과 같아야 합니다."

        # 2. 슬라이딩 윈도우(Sliding Window)로 데이터 생성
        # 전체 토큰 리스트를 훑으며 max_length 길이만큼 잘라냅니다.
        # stride만큼 건너뛰며 반복합니다.
        for i in range(0, len(token_ids) - max_length, stride):
            # 입력 청크: 현재 위치(i)부터 max_length만큼 가져옵니다.
            # region [input 청크 생성]
            # TODO: 입력 청크 슬라이싱 구간을 채우세요.
            # 힌트: 현재 위치 i에서 시작해 max_length 길이만큼 자르면 됩니다.
            # 힌트: 시작은 i, 끝은 i + max_length입니다.
            # 예시: token_ids=[1,2,3,4,5], i=1, max_length=3 -> [2,3,4]
            input_chunk = token_ids[???? : ???? + max_length]
            # endregion
              
            # 타겟 청크: 입력보다 1칸 뒤의 위치(i+1)부터 가져옵니다.
            # GPT는 '다음 단어'를 맞추는 모델이므로, 정답은 입력보다 한 칸씩 뒤로 밀려있어야 합니다.
            # region [target 청크 생성]
            # TODO: 타깃 청크 슬라이싱 구간을 채우세요.
            # 힌트: 입력보다 한 칸 뒤에서 시작하도록 i+1 기준을 사용하세요.
            # 힌트: 시작과 끝 모두 i 기준으로 +1 이동합니다.
            # 예시: input=[2,3,4]이면 target=[3,4,5]
            target_chunk = token_ids[???? + ???? : ???? + max_length + ????]
            # endregion

            # 추출한 데이터를 텐서(Tensor)로 변환하여 리스트에 저장합니다.
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # 데이터셋의 총 샘플(청크) 개수를 반환합니다.
        return len(self.input_ids)

    def __getitem__(self, idx):
        # DataLoader가 데이터를 요청할 때 호출됩니다.
        # 해당 인덱스(idx)의 입력과 정답 쌍을 반환합니다.
        # TODO: __getitem__에서 같은 idx의 입력/타깃 쌍을 반환하세요.
        # 힌트: 두 리스트에서 동일한 idx를 사용해 페어를 맞추세요.
        # 힌트: self.input_ids와 self.target_ids에 동일한 idx를 사용합니다.
        # 예시: return self.input_ids[idx], self.target_ids[idx]
        return self.input_ids[????], self.target_ids[????]

In [ ]:
##중요 : 아래 함수는 DataLoader를 생성하는 헬퍼 함수입니다.
import tiktoken
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # 토크나이저를 초기화합니다.
    tokenizer = tiktoken.get_encoding("gpt2")

    # 데이터셋을 만듭니다.
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 데이터 로더를 만듭니다.
    # region [DataLoader 생성]
    # TODO: DataLoader의 첫 번째 인자를 채우세요.
    # 힌트: 위에서 생성한 dataset 객체를 그대로 전달하면 됩니다.
    # 힌트: 위에서 생성한 GPTDatasetV1 인스턴스를 그대로 사용하면 됩니다.
    # 예시: loader = DataLoader(dataset, batch_size=2, shuffle=True)
    dataloader = DataLoader(
        ????,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    # endregion
    
    return dataloader

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)

first_batch = next(data_iter)
print(first_batch)

second_batch = next(data_iter)
print(second_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
# region [임베딩 레이어 생성]
# TODO: Embedding 생성자 인자를 채우세요.
# 힌트: 첫 번째는 vocab_size, 두 번째는 output_dim입니다.
# 힌트: 단어장 크기는 vocab_size, 임베딩 차원은 output_dim입니다.
# 예시: emb = torch.nn.Embedding(10000, 256)
embedding_layer = torch.nn.Embedding(????, ????)
# endregion

print(embedding_layer.weight)

# region [임베딩 레이어 적용]
# TODO: 임베딩 레이어 입력 변수를 채우세요.
# 힌트: 토큰 ID 텐서인 input_ids를 전달하면 됩니다.
# 힌트: input_ids 변수를 그대로 인자로 전달하면 됩니다.
# 예시: vecs = emb(torch.tensor([1, 5, 9]))
token_embeddings = embedding_layer(????)
# endregion
print(token_embeddings)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

context_length = max_length
# region [포지셔널 임베딩 레이어 생성]
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
# endregion
# region [포지셔널 임베딩 적용]
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
# endregion
print(pos_embeddings.shape)
# region [토큰 임베딩과 포지셔널 임베딩 합산]
input_embeddings = token_embeddings + pos_embeddings
# endregion
print(input_embeddings.shape)


torch.Size([8, 4, 256])
torch.Size([4, 256])
torch.Size([8, 4, 256])


## Tokenizer 비교

- 아래 결과를 보면 한국어는 Exaone, 중국어는 Qwen의 Tokenizer 성능이 좋습니다

In [9]:
from transformers import AutoTokenizer
# Tokenizer 로드 (로컬 또는 온라인에서)
print("Tokenizer 로드 중...")
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
exaone = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-4.0-1.2B")

# 테스트할 텍스트
texts = [
    "안녕하세요, 반갑습니다!",
    "Hello, nice to meet you!",
    "인공지능 기술이 발전하고 있습니다.",
    "Hello! This is an English tokenizer test.",
    "こんにちは！日本語のテストです。",
    "人工智能技术正在快速发展。",
    "The quick brown fox jumps over the lazy dog. 빠른 갈색 여우가 게으른 개를 뛰어넘습니다.",
    "파이썬 프로그래밍은 매우 유용합니다. Python programming is very useful.",
    "123456789 !@#$%^&*() 특수문자",
]

print("\n" + "="*60)
for text in texts:
    print(f"\n텍스트: {text}")
    print("-"*60)
    
    # Qwen3
    qwen_tokens = qwen.encode(text, add_special_tokens=True)
    qwen_token_strs = [qwen.decode([tok]) for tok in qwen_tokens[:10]]
    print(f"Qwen3:  {len(qwen_tokens)}개 토큰")
    print(f"        {qwen_token_strs}")
    
    # EXAONE
    exaone_tokens = exaone.encode(text, add_special_tokens=True)
    exaone_token_strs = [exaone.decode([tok]) for tok in exaone_tokens[:10]]
    print(f"EXAONE: {len(exaone_tokens)}개 토큰")
    print(f"        {exaone_token_strs}")
    
    # 비교
    diff = len(qwen_tokens) - len(exaone_tokens)
    if diff > 0:
        print(f"→ EXAONE이 {diff}개 더 효율적")
    elif diff < 0:
        print(f"→ Qwen3가 {abs(diff)}개 더 효율적")
    else:
        print(f"→ 동일")

    # Special tokens 비교
print("\n" + "=" * 80)
print("4. Special Tokens 비교")
print("=" * 80)

print(f"\n[Qwen3 Special Tokens]")
for token_name in ['bos_token', 'eos_token', 'unk_token', 'pad_token']:
    token = getattr(qwen, token_name, None)
    if token:
        token_id = getattr(qwen, f"{token_name}_id", None)
        print(f"  {token_name}: '{token}' (ID: {token_id})")

print(f"\n[EXAONE Special Tokens]")
for token_name in ['bos_token', 'eos_token', 'unk_token', 'pad_token']:
    token = getattr(exaone, token_name, None)
    if token:
        token_id = getattr(exaone, f"{token_name}_id", None)
        print(f"  {token_name}: '{token}' (ID: {token_id})")

Tokenizer 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]



텍스트: 안녕하세요, 반갑습니다!
------------------------------------------------------------
Qwen3:  8개 토큰
        ['안', '녕', '하세요', ',', ' 반', '갑', '습니다', '!']
EXAONE: 7개 토큰
        ['안녕', '하', '세요', ',', ' 반갑', '습니다', '!']
→ EXAONE이 1개 더 효율적

텍스트: Hello, nice to meet you!
------------------------------------------------------------
Qwen3:  7개 토큰
        ['Hello', ',', ' nice', ' to', ' meet', ' you', '!']
EXAONE: 7개 토큰
        ['Hello', ',', ' nice', ' to', ' meet', ' you', '!']
→ 동일

텍스트: 인공지능 기술이 발전하고 있습니다.
------------------------------------------------------------
Qwen3:  12개 토큰
        ['인', '공', '지', '능', ' 기', '술', '이', ' 발', '전', '하고']
EXAONE: 9개 토큰
        ['인공', '지능', ' 기술', '이', ' 발전', '하고', ' 있', '습니다', '.']
→ EXAONE이 3개 더 효율적

텍스트: Hello! This is an English tokenizer test.
------------------------------------------------------------
Qwen3:  9개 토큰
        ['Hello', '!', ' This', ' is', ' an', ' English', ' tokenizer', ' test', '.']
EXAONE: 9개 토큰
        ['Hello', '!', ' This', ' is

## Hugging Face 모델 파일 구조 설명

Hugging Face Hub에 업로드된 파일들은 **Transformer 기반 언어 모델(LLM)**을 구성하는 핵심 요소들입니다. 크게 **설계도(Config)**, **가중치(Weights)**, **토크나이저(Tokenizer)** 세 그룹으로 나뉩니다.

---

### 1. 모델 아키텍처 및 설정 (설계도)

* **`config.json`**
    * **의미:** 모델의 **구조와 설계도**가 담긴 가장 중요한 설정 파일입니다.
    * **내용:** 모델의 종류(Llama, BERT 등), 레이어(Layer) 수, 히든 사이즈, 어텐션 헤드 수 등을 정의하여 빈 모델(Skeleton)을 만듭니다.

* **`generation_config.json`**
    * **의미:** 텍스트 **생성(Generation) 시 사용할 기본 옵션** 파일입니다.
    * **내용:** `max_length`(최대 길이), `temperature`(창의성), `top_p`, `eos_token_id`(종료 토큰) 등의 기본 설정값이 들어있습니다.

### 2. 모델 가중치 (학습된 뇌)

* **`model.safetensors`**
    * **의미:** 실제 **학습된 파라미터(가중치)**가 저장된 파일입니다. (가장 용량이 큼)
    * **특징:**
        * 기존 PyTorch의 `.bin` 파일보다 로딩 속도가 빠릅니다.
        * 보안 취약점(악성 코드 실행 등)을 해결한 안전한 형식입니다.

### 3. 토크나이저 (언어 처리기)
사람의 언어(Text)를 모델이 이해하는 숫자(Token ID)로 변환하는 규칙들입니다.

* **`vocab.json`**
    * **의미:** **단어장(Vocabulary)** 파일입니다. 단어와 그에 대응하는 숫자 ID가 매핑되어 있습니다.
* **`merges.txt`**
    * **의미:** **BPE 병합 규칙** 파일입니다. 알파벳 조각들이 어떻게 합쳐져서 단어가 되는지 정의합니다.
* **`tokenizer.json`**
    * **의미:** `vocab.json`과 `merges.txt` 등을 하나로 통합하여 최적화한 파일로, 로딩 속도가 매우 빠릅니다.
* **`tokenizer_config.json`**
    * **의미:** 토크나이저 설정 파일입니다. 어떤 클래스를 쓸지, 특수 토큰(`<s>`, `<pad>` 등)은 무엇인지 정의합니다.

---

### 요약

| 파일명 | 비유 | 역할 |
| :--- | :--- | :--- |
| `config.json` | **설계도** | 모델 구조 정의 |
| `model.safetensors` | **뇌** | 학습된 지식(가중치) 저장 |
| `tokenizer.json` | **번역기** | 텍스트 ↔ 숫자 변환 |
| `generation_config.json` | **말하기 습관** | 생성 옵션 설정 |